# Course 12: Introduction to Large Language Models
## GCP Machine Learning Engineer Certification

This notebook covers hands-on exercises for LLM concepts on Google Cloud:
- Compare Gemini Pro vs Flash: latency, quality, cost
- Fine-tuning concepts with Vertex AI (SDK code, commented for cost)
- Prompt engineering techniques: zero-shot, few-shot, system instructions
- Token counting and cost estimation
- Evaluation: compare model outputs on same prompts
- Embeddings: generate and compare text embeddings

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
# Install the Vertex AI SDK
!pip install -q google-cloud-aiplatform>=1.38.0 google-generativeai>=0.3.0

In [ ]:
# Authenticate (Colab)
from google.colab import auth
auth.authenticate_user()

# Set your project and location
PROJECT_ID = "your-project-id"  # <-- Replace with your GCP project ID
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Vertex AI initialized: project={PROJECT_ID}, location={LOCATION}")

In [ ]:
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part
import time

print("Imports successful.")

---
## 2. Compare Gemini Pro vs Flash: Latency, Quality, Cost

In [ ]:
# Initialize both models
model_pro = GenerativeModel("gemini-1.5-pro")
model_flash = GenerativeModel("gemini-1.5-flash")

# Test prompts of varying complexity
test_prompts = {
    "simple": "What is the capital of France?",
    "moderate": "Explain the difference between supervised and unsupervised learning in 3 sentences.",
    "complex": """A company has 10TB of customer support tickets spanning 5 years.
They want to automatically categorize tickets, extract key entities, and generate
summary reports. Design a complete ML pipeline on GCP, including data processing,
model selection, training, deployment, and monitoring. Justify each choice.""",
}

print("Models initialized. Ready for comparison.")

In [ ]:
# Compare latency and output quality
results = {}

for complexity, prompt in test_prompts.items():
    print(f"\n{'='*60}")
    print(f"Complexity: {complexity.upper()}")
    print(f"Prompt: {prompt[:80]}..." if len(prompt) > 80 else f"Prompt: {prompt}")
    print(f"{'='*60}")
    
    for model_name, model in [("Pro", model_pro), ("Flash", model_flash)]:
        start_time = time.time()
        response = model.generate_content(prompt)
        latency = time.time() - start_time
        
        token_count = response.usage_metadata.total_token_count
        output_tokens = response.usage_metadata.candidates_token_count
        
        results[f"{complexity}_{model_name}"] = {
            "latency": latency,
            "total_tokens": token_count,
            "output_tokens": output_tokens,
            "text": response.text,
        }
        
        print(f"\n--- {model_name} ---")
        print(f"Latency: {latency:.2f}s")
        print(f"Output tokens: {output_tokens}")
        print(f"Response preview: {response.text[:200]}...")

In [ ]:
# Cost estimation (approximate pricing as of 2024)
# Gemini 1.5 Pro: $3.50/M input tokens, $10.50/M output tokens (128K context)
# Gemini 1.5 Flash: $0.075/M input tokens, $0.30/M output tokens (128K context)

PRICING = {
    "Pro":   {"input": 3.50 / 1_000_000, "output": 10.50 / 1_000_000},
    "Flash": {"input": 0.075 / 1_000_000, "output": 0.30 / 1_000_000},
}

print("=== Cost Comparison ===")
print(f"{'Task':<12} {'Model':<8} {'Latency':>9} {'Tokens':>8} {'Cost':>12}")
print("-" * 55)

for complexity in test_prompts:
    for model_name in ["Pro", "Flash"]:
        key = f"{complexity}_{model_name}"
        r = results[key]
        
        # Estimate input tokens from total - output
        input_tokens = r["total_tokens"] - r["output_tokens"]
        cost = (input_tokens * PRICING[model_name]["input"] + 
                r["output_tokens"] * PRICING[model_name]["output"])
        
        print(f"{complexity:<12} {model_name:<8} {r['latency']:>8.2f}s {r['total_tokens']:>7} ${cost:>10.6f}")

print("\nNote: Flash is typically 10-50x cheaper than Pro per request.")

### Key Takeaways

- **Gemini Pro**: Higher quality for complex reasoning tasks, but slower and more expensive
- **Gemini Flash**: Much faster and cheaper, with quality sufficient for most production tasks
- **Exam tip**: Always start with Flash for cost-sensitive production workloads; upgrade to Pro only when quality demands it

---
## 3. Fine-Tuning Concepts with Vertex AI

Fine-tuning on Vertex AI incurs significant compute costs. The code below demonstrates the SDK
interface with comments explaining each step. **Do not run the actual tuning job unless you have
budget allocated.**

In [ ]:
# === FINE-TUNING WITH VERTEX AI (commented to prevent accidental cost) ===

# Step 1: Prepare training data in JSONL format
# Each line is a JSON object with "messages" array
TRAINING_DATA_EXAMPLE = """
{"messages": [{"role": "system", "content": "You are a GCP expert."}, {"role": "user", "content": "What is BigQuery?"}, {"role": "model", "content": "BigQuery is Google Cloud's fully managed, serverless data warehouse..."}]}
{"messages": [{"role": "system", "content": "You are a GCP expert."}, {"role": "user", "content": "Explain Vertex AI pipelines."}, {"role": "model", "content": "Vertex AI Pipelines is a managed service for orchestrating ML workflows..."}]}
"""

print("Training data format (JSONL):")
print(TRAINING_DATA_EXAMPLE)

# Step 2: Upload training data to GCS
# gsutil cp training_data.jsonl gs://your-bucket/tuning/training_data.jsonl

print("Upload training data to GCS bucket before tuning.")

In [ ]:
# Step 3: Launch a tuning job (COMMENTED - will incur cost if uncommented)

# from vertexai.generative_models import GenerativeModel
#
# # Supervised tuning of Gemini
# tuning_job = GenerativeModel.tune_model(
#     source_model="gemini-1.5-flash",
#     training_data="gs://your-bucket/tuning/training_data.jsonl",
#     # Optional: validation data
#     # validation_data="gs://your-bucket/tuning/validation_data.jsonl",
#     tuned_model_display_name="my-tuned-gemini-flash",
#     epochs=3,            # Number of training epochs (1-10)
#     learning_rate=1e-4,  # Learning rate (typically 1e-5 to 1e-3)
# )
#
# # Monitor the tuning job
# print(f"Tuning job: {tuning_job.resource_name}")
# tuning_job.wait()  # This blocks until the job completes
#
# # Use the tuned model
# tuned_model = GenerativeModel(tuning_job.tuned_model_endpoint_name)
# response = tuned_model.generate_content("What is Cloud Spanner?")
# print(response.text)

print("=== Fine-Tuning Code (Reference Only) ===")
print("The code above demonstrates the Vertex AI tuning SDK interface.")
print("Key parameters:")
print("  - source_model: Base model to fine-tune (gemini-1.5-flash, gemini-1.5-pro)")
print("  - training_data: GCS path to JSONL training data")
print("  - epochs: 1-10 (start with 3, increase if underfitting)")
print("  - learning_rate: 1e-5 to 1e-3 (start with 1e-4)")
print("\nVertex AI handles LoRA/PEFT configuration automatically.")

---
## 4. Prompt Engineering Techniques

In [ ]:
# Zero-shot with system instructions
model_with_system = GenerativeModel(
    "gemini-1.5-flash",
    system_instruction="""You are a senior GCP solutions architect. 
Always structure your responses with:
1. A one-sentence summary
2. Key technical details
3. A practical recommendation
Keep responses under 150 words."""
)

response = model_with_system.generate_content(
    "When should I use Cloud Run vs GKE for serving an ML model?"
)

print("=== System Instructions ===")
print(response.text)

In [ ]:
# Few-shot with structured output
few_shot_prompt = """Extract GCP services mentioned in the text. Return as a JSON array.

Text: "We used BigQuery for analytics and Cloud Storage for the data lake."
Services: ["BigQuery", "Cloud Storage"]

Text: "The pipeline runs on Dataflow, stores results in Firestore, and triggers via Pub/Sub."
Services: ["Dataflow", "Firestore", "Pub/Sub"]

Text: "Our ML models are trained on Vertex AI, served via Cloud Run, and monitored with Cloud Monitoring."
Services:"""

response = model_flash.generate_content(few_shot_prompt)
print("=== Few-Shot Structured Output ===")
print(response.text)

In [ ]:
# Compare zero-shot vs few-shot accuracy on a classification task
test_cases = [
    ("The model's accuracy dropped from 95% to 60% after deploying to production.", "Data Drift"),
    ("Training takes 48 hours on a single GPU but we need results in 2 hours.", "Compute Scaling"),
    ("Users report the chatbot gives different answers to the same question.", "Non-determinism"),
]

# Zero-shot
print("=== Zero-Shot Classification ===")
for text, expected in test_cases:
    response = model_flash.generate_content(
        f"Classify this ML issue into one category: Data Drift, Compute Scaling, Non-determinism, or Other.\n\nIssue: {text}\nCategory:"
    )
    print(f"  Issue: {text[:60]}...")
    print(f"  Expected: {expected} | Got: {response.text.strip()}")

print("\n=== Few-Shot Classification ===")
few_shot_prefix = """Classify each ML issue into exactly one category: Data Drift, Compute Scaling, Non-determinism, or Other.

Issue: "Feature distributions in production differ significantly from training data."
Category: Data Drift

Issue: "We need to process 1M inference requests per second but our GPU can only handle 100."
Category: Compute Scaling

Issue: "The model returns slightly different predictions for identical inputs across requests."
Category: Non-determinism

"""

for text, expected in test_cases:
    response = model_flash.generate_content(
        few_shot_prefix + f"Issue: \"{text}\"\nCategory:"
    )
    print(f"  Issue: {text[:60]}...")
    print(f"  Expected: {expected} | Got: {response.text.strip()}")

---
## 5. Token Counting and Cost Estimation

In [ ]:
# Count tokens before sending a request (useful for cost estimation)

test_texts = [
    "Hello, world!",
    "Explain the transformer architecture in detail, including multi-head attention, positional encoding, feed-forward networks, and residual connections.",
    "A" * 1000,  # 1000 'A' characters
    open("/dev/null", "r").read() if False else "This is a placeholder for a long document.",
]

print("=== Token Counting ===")
for text in test_texts:
    token_count = model_flash.count_tokens(text)
    chars = len(text)
    ratio = chars / token_count.total_tokens if token_count.total_tokens > 0 else 0
    
    print(f"  Text: {text[:50]}{'...' if len(text) > 50 else ''}")
    print(f"  Characters: {chars} | Tokens: {token_count.total_tokens} | Chars/Token: {ratio:.1f}")
    print()

In [ ]:
# Cost estimation function

def estimate_cost(input_tokens, output_tokens, model_name="flash"):
    """Estimate the cost of a Gemini API call.
    
    Args:
        input_tokens: Number of input tokens
        output_tokens: Number of output tokens  
        model_name: 'pro' or 'flash'
    
    Returns:
        Estimated cost in USD
    """
    pricing = {
        "pro":   {"input": 3.50 / 1_000_000, "output": 10.50 / 1_000_000},
        "flash": {"input": 0.075 / 1_000_000, "output": 0.30 / 1_000_000},
    }
    
    p = pricing[model_name]
    return input_tokens * p["input"] + output_tokens * p["output"]


# Estimate costs for different usage scenarios
scenarios = [
    ("Single chat turn", 100, 200),
    ("Document summary (10 pages)", 5000, 500),
    ("Long document analysis", 100000, 2000),
    ("Daily batch: 10K requests", 100 * 10000, 200 * 10000),
]

print("=== Cost Estimation ===")
print(f"{'Scenario':<30} {'Input':>8} {'Output':>8} {'Pro Cost':>12} {'Flash Cost':>12} {'Savings':>8}")
print("-" * 82)

for name, input_tok, output_tok in scenarios:
    cost_pro = estimate_cost(input_tok, output_tok, "pro")
    cost_flash = estimate_cost(input_tok, output_tok, "flash")
    savings = (1 - cost_flash / cost_pro) * 100 if cost_pro > 0 else 0
    
    print(f"{name:<30} {input_tok:>8} {output_tok:>8} ${cost_pro:>10.4f} ${cost_flash:>10.4f} {savings:>6.0f}%")

---
## 6. Evaluation: Compare Model Outputs

In [ ]:
# Evaluate models on the same set of prompts
# We use a simple evaluation rubric: accuracy, completeness, conciseness

eval_prompts = [
    {
        "prompt": "What are the three V's of big data?",
        "reference": "Volume, Velocity, Variety",
        "type": "factual"
    },
    {
        "prompt": "Write a Python function to calculate the factorial of a number.",
        "reference": "def factorial(n): return 1 if n <= 1 else n * factorial(n-1)",
        "type": "code"
    },
    {
        "prompt": "Summarize the CAP theorem in exactly 2 sentences.",
        "reference": "The CAP theorem states that a distributed system can provide at most two of three guarantees: Consistency, Availability, and Partition tolerance. Since network partitions are inevitable, systems must choose between consistency and availability during a partition.",
        "type": "summarization"
    },
]

print("=== Model Evaluation ===")
for eval_item in eval_prompts:
    print(f"\n--- {eval_item['type'].upper()} ---")
    print(f"Prompt: {eval_item['prompt']}")
    print(f"Reference: {eval_item['reference'][:100]}...")
    
    for model_name, model in [("Pro", model_pro), ("Flash", model_flash)]:
        response = model.generate_content(
            eval_item["prompt"],
            generation_config=GenerationConfig(temperature=0, max_output_tokens=200)
        )
        print(f"  {model_name}: {response.text.strip()[:150]}")

In [ ]:
# LLM-as-Judge: Use Gemini Pro to evaluate Flash outputs

judge_prompt_template = """You are an expert evaluator. Rate the following model response on a scale of 1-5 
for each criterion. Return ONLY a JSON object with scores.

Criteria:
- accuracy: Is the information factually correct? (1=wrong, 5=perfectly correct)
- completeness: Does it fully answer the question? (1=incomplete, 5=comprehensive)
- conciseness: Is it appropriately concise? (1=too verbose/short, 5=perfect length)

Question: {question}
Reference answer: {reference}
Model response: {response}

JSON scores:"""

print("=== LLM-as-Judge Evaluation ===")
for eval_item in eval_prompts:
    # Get Flash response
    flash_response = model_flash.generate_content(
        eval_item["prompt"],
        generation_config=GenerationConfig(temperature=0, max_output_tokens=200)
    )
    
    # Judge with Pro
    judge_prompt = judge_prompt_template.format(
        question=eval_item["prompt"],
        reference=eval_item["reference"],
        response=flash_response.text.strip()
    )
    
    judgment = model_pro.generate_content(
        judge_prompt,
        generation_config=GenerationConfig(temperature=0, max_output_tokens=100)
    )
    
    print(f"\nTask: {eval_item['type']}")
    print(f"Flash response: {flash_response.text.strip()[:100]}...")
    print(f"Judge scores: {judgment.text.strip()}")

---
## 7. Embeddings: Generate and Compare Text Embeddings

In [ ]:
from vertexai.language_models import TextEmbeddingModel
import numpy as np

# Initialize the embedding model
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

print(f"Embedding model loaded: text-embedding-004")

In [ ]:
# Generate embeddings for sample texts
texts = [
    "How do I train a machine learning model on Google Cloud?",
    "What are the steps to build an ML model using Vertex AI?",
    "How do I bake a chocolate cake?",
    "Vertex AI provides tools for training and deploying ML models.",
    "The weather in Paris is beautiful in spring.",
]

# Get embeddings
embeddings = embedding_model.get_embeddings(texts)

# Convert to numpy arrays
embedding_vectors = np.array([e.values for e in embeddings])

print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimension: {len(embeddings[0].values)}")
print(f"First embedding (first 10 values): {embeddings[0].values[:10]}")

In [ ]:
# Compute cosine similarity between all pairs
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=== Cosine Similarity Matrix ===")
print(f"{'':>5}", end="")
for i in range(len(texts)):
    print(f"  T{i+1:>2}", end="")
print()

for i in range(len(texts)):
    print(f"T{i+1:>2} |", end="")
    for j in range(len(texts)):
        sim = cosine_similarity(embedding_vectors[i], embedding_vectors[j])
        print(f" {sim:.2f}", end="")
    print(f"  <- {texts[i][:45]}..." if len(texts[i]) > 45 else f"  <- {texts[i]}")

print("\nExpected: T1-T2 (similar topic) should have high similarity.")
print("T1-T3 (different topics) should have low similarity.")
print("T1-T4 (related GCP/ML topic) should have moderate-high similarity.")

In [ ]:
# Semantic search example: find the most relevant text for a query

query = "I want to deploy a model to production on GCP"
query_embedding = embedding_model.get_embeddings([query])[0].values
query_vector = np.array(query_embedding)

print(f"=== Semantic Search ===")
print(f"Query: {query}\n")

# Compute similarities and rank
similarities = []
for i, text in enumerate(texts):
    sim = cosine_similarity(query_vector, embedding_vectors[i])
    similarities.append((sim, text))

# Sort by similarity (descending)
similarities.sort(reverse=True)

print("Results (ranked by relevance):")
for rank, (sim, text) in enumerate(similarities, 1):
    print(f"  {rank}. [{sim:.3f}] {text}")

---
## Summary

In this notebook, you learned:

1. **Model Comparison**: How to benchmark Gemini Pro vs Flash on latency, quality, and cost
2. **Fine-Tuning**: The Vertex AI SDK interface for supervised tuning (LoRA applied automatically)
3. **Prompt Engineering**: System instructions, few-shot prompting, and structured outputs
4. **Token Counting**: How to count tokens and estimate costs before making API calls
5. **Evaluation**: Comparing model outputs and using LLM-as-Judge for automated evaluation
6. **Embeddings**: Generating text embeddings, computing similarity, and building semantic search

**Key Exam Concepts**:
- Flash for speed/cost, Pro for quality, Gemma for self-hosting
- Prompt engineering first, fine-tuning second
- Token counting is essential for cost management
- Embeddings enable semantic search and RAG pipelines